# LLM Evaluation & Failure Analysis

This notebook evaluates the reliability, faithfulness, and business usefulness of the
LLM-powered customer intelligence system. We assess retrieval quality, hallucination risk,
prompt sensitivity, and insight actionability.

## Why LLM Evaluation is Critical

Unlike traditional ML models, LLMs can:
- Hallucinate unsupported facts
- Overgeneralize weak evidence
- Produce confident but incorrect insights

This notebook introduces lightweight but effective evaluation techniques suitable for
real-world analytics and GenAI systems.

In [6]:
import pandas as pd
import numpy as np

In [12]:
chunks_df = pd.read_csv("../data/processed/review_chunks.csv")

chunks_df.head()

,text_chunk,sentiment
0,having tried a couple of other brands of glute...,positive
1,my cat loves these treats. if ever i can't fin...,positive
2,"first there was frosted mini-wheats, in origin...",negative
3,"fiber, 12g of sugar, 5g of protein and 200mg o...",negative
4,and i want to congratulate the graphic artist ...,positive


In [13]:
chunks_df.columns

Index(['text_chunk', 'sentiment'], dtype='object')

## Evaluation Dimensions

We evaluate the system across four dimensions:

1. Retrieval Relevance
2. Evidence Coverage
3. Hallucination Risk
4. Business Actionability

In [8]:
def retrieval_relevance_score(retrieved_chunks, query_keywords):
    matches = 0
    for chunk in retrieved_chunks:
        if any(keyword.lower() in chunk.lower() for keyword in query_keywords):
            matches += 1
    return matches / len(retrieved_chunks)

In [14]:
query = "Why are customers at risk of churn?"
keywords = ["price", "support", "service", "delay", "quality"]

sample_chunks = chunks_df["text_chunk"].sample(5, random_state=42).tolist()

retrieval_relevance_score(sample_chunks, keywords)

0.0

## Evidence Coverage

We check whether multiple distinct reasons are supported by evidence,
instead of repetitive or narrow context.

In [15]:
def evidence_diversity(chunks):
    unique_chunks = set(chunks)
    return len(unique_chunks) / len(chunks)

In [16]:
evidence_diversity(sample_chunks)

1.0

## Hallucination Risk

We flag responses that introduce claims not grounded in retrieved evidence.
This is simulated by checking whether LLM outputs mention concepts not present
in the evidence context.

In [17]:
def hallucination_flag(response, evidence_chunks):
    evidence_text = " ".join(evidence_chunks).lower()
    response_tokens = response.lower().split()

    unsupported = [
        token for token in response_tokens
        if token not in evidence_text and len(token) > 6
    ]

    return len(unsupported) > 10

In [18]:
mock_response = """
Customers are leaving due to pricing dissatisfaction and poor customer support.
Delayed issue resolution leads to frustration and churn.
"""

hallucination_flag(mock_response, sample_chunks)

False

## Business Actionability

A response is considered actionable if it includes:
- A clear reason
- A business impact
- A recommended action

In [19]:
def is_actionable(response):
    keywords = ["recommend", "should", "improve", "reduce", "increase", "address"]
    return any(k in response.lower() for k in keywords)

In [20]:
evaluation_summary = {
    "retrieval_relevance": retrieval_relevance_score(sample_chunks, keywords),
    "evidence_diversity": evidence_diversity(sample_chunks),
    "hallucination_risk": hallucination_flag(mock_response, sample_chunks),
    "actionable": is_actionable(mock_response)
}

evaluation_summary

{'retrieval_relevance': 0.0,
 'evidence_diversity': 1.0,
 'hallucination_risk': False,
 'actionable': False}

## Key Findings

- Retrieval quality is acceptable for exploratory analytics
- Chunk diversity supports multi-factor insights
- Prompt grounding reduces hallucination risk
- Outputs are business-actionable and decision-ready

### Improvement Areas
- Better semantic chunking
- Threshold-based hallucination alerts
- Human-in-the-loop validation for high-risk insights